# Support Ticket Classification: Word Counts + NN and DistilBERT

Download [support_tickets.csv](sample_data/support_tickets.csv) and keep it in `sample_data/` beside this notebook, or change `TICKET_CSV` below. Run cells from top to bottom. Both models use the same supplied train/validation/test splits.

## 1. Define the Task and Inspect the Data

Compare **raw ticket → word counts → learned NN features → category** with **raw ticket → token IDs → embeddings → contextual encoder features → classification head → category**.

This extends the count-feature idea in Sessions 5–6. Here a small fully connected neural network replaces logistic regression. Each ticket has one category, so this is single-label multiclass classification.



[Download the supplied ticket CSV](sample_data/support_tickets.csv).

The supplied CSV contains 80 fictional teaching tickets, with 20 in each class and explicit train/validation/test splits. It is deliberately small and balanced. It cannot estimate real service-desk performance, and short synthetic wording can make classification easier than realistic tickets.

| Category | Routing rule | Example |
|---|---|---|
| Account access | Credentials, locked accounts and authentication | My account is locked. |
| Hardware | Physical device faults | The laptop screen is cracked. |
| Software | Application installation, crashes and behaviour | The spreadsheet application crashes. |
| Network | Connectivity and network access | The campus Wi-Fi disconnects. |

A real ticket can involve several issues. This exercise assigns one primary category; ambiguous cases should be reviewed rather than forcing an unjustified label.


In [ ]:
import pandas as pd

TICKET_CSV = "sample_data/support_tickets.csv"
tickets = pd.read_csv(TICKET_CSV)  # Open the supplied labelled data when first used.
print(tickets.groupby(["split", "category"]).size().unstack(fill_value=0))
print(tickets.head(8).to_string(index=False))
assert not tickets["text"].str.lower().duplicated().any(), "Check duplicate ticket text before training."

class_names = ["Account access", "Hardware", "Software", "Network"]
label_to_id = {name: number for number, name in enumerate(class_names)}
id_to_label = {number: name for name, number in label_to_id.items()}
tickets["label"] = tickets["category"].map(label_to_id)
train_tickets = tickets.loc[tickets["split"] == "train"].copy()
validation_tickets = tickets.loc[tickets["split"] == "validation"].copy()
test_tickets = tickets.loc[tickets["split"] == "test"].copy()


## 2. Word Counts + a Neural Network

`CountVectorizer` builds a vocabulary from **training text only**. Each ticket becomes a vector of word occurrence counts. The vocabulary and counts are fixed; a hidden layer learns combinations of those counts, and an output layer scores the four categories.

Like our CNN training loop, training uses forward prediction, loss, gradients, optimiser updates and gradient reset. Cross-entropy uses one integer category per ticket. Select the NN weights using validation loss; keep the test set for the final comparison.

This small example converts the count matrix to a dense tensor. Large vocabularies need more memory. Counts ignore word order and context; changing word order alone leaves the vector unchanged.

If needed, install the libraries for this first method in your terminal:

```bash
python -m pip install "torch>=2.6,<3" pandas scikit-learn matplotlib
```

In [ ]:
import torch
from torch import nn
from sklearn.feature_extraction.text import CountVectorizer

SEED = 42
torch.manual_seed(SEED)  # Reproduce the initial NN weights.
count_vectorizer = CountVectorizer(lowercase=True, ngram_range=(1, 1))
# Fit the vocabulary on training tickets; transform validation with that same vocabulary.
train_counts = count_vectorizer.fit_transform(train_tickets["text"])
validation_counts = count_vectorizer.transform(validation_tickets["text"])
X_count_train = torch.tensor(train_counts.toarray(), dtype=torch.float32)
X_count_validation = torch.tensor(validation_counts.toarray(), dtype=torch.float32)
y_count_train = torch.tensor(train_tickets["label"].to_numpy(), dtype=torch.long)
y_count_validation = torch.tensor(validation_tickets["label"].to_numpy(), dtype=torch.long)

print("Vocabulary size:", len(count_vectorizer.vocabulary_))
print("Training input shape:", tuple(X_count_train.shape))
print("First ticket:", train_tickets.iloc[0]["text"])
first_counts = train_counts[0].toarray()[0]
print([(word, int(first_counts[index])) for index, word in enumerate(count_vectorizer.get_feature_names_out()) if first_counts[index] > 0])

In [ ]:
import copy

COUNT_HIDDEN_UNITS = 32
COUNT_EPOCHS = 100
COUNT_LEARNING_RATE = 0.01

def build_count_nn(input_size, hidden_size, output_size):
    # The hidden layer learns feature combinations; the last layer returns class logits.
    return nn.Sequential(
        nn.Linear(input_size, hidden_size), nn.ReLU(),
        nn.Linear(hidden_size, output_size),
    )

count_model = build_count_nn(X_count_train.shape[1], COUNT_HIDDEN_UNITS, len(class_names))
count_loss = nn.CrossEntropyLoss()  # One correct category for each ticket.
count_optimiser = torch.optim.Adam(count_model.parameters(), lr=COUNT_LEARNING_RATE)
best_validation_loss = float("inf")
best_count_weights = copy.deepcopy(count_model.state_dict())
count_history = []

for epoch in range(COUNT_EPOCHS):
    count_model.train()
    count_optimiser.zero_grad()  # Clear gradients from the previous update.
    training_loss = count_loss(count_model(X_count_train), y_count_train)
    training_loss.backward()
    count_optimiser.step()
    count_model.eval()
    with torch.no_grad():
        validation_loss = count_loss(count_model(X_count_validation), y_count_validation)
    count_history.append({"epoch": epoch + 1, "train_loss": training_loss.item(), "validation_loss": validation_loss.item()})
    if validation_loss.item() < best_validation_loss:
        best_validation_loss = validation_loss.item()
        best_count_weights = copy.deepcopy(count_model.state_dict())

# Restore the weights selected using validation data, rather than assuming the last epoch is best.
count_model.load_state_dict(best_count_weights)
count_model.eval()
print("Best count-NN validation loss:", round(best_validation_loss, 4))
print(pd.DataFrame(count_history).iloc[::20].to_string(index=False))

## 3. Prepare Inputs and Add the Classification Head

Each ticket label is one integer identifying its category. DistilBERT supplies pretrained language features, and a newly initialised classification head supplies four task scores. The initialisation warning for the new head is expected: its categories have not been learned yet.

Padding makes batch lengths compatible; its attention mask marks real tokens. Truncation limits input length and can remove useful details from long tickets.


Before using Transformers, install these packages in your terminal, then restart the notebook kernel. The practical uses the Transformers 4.57 API. Run from the course folder; a GPU is recommended for training.

```bash
python -m pip install "torch>=2.6,<3" "transformers==4.57.1" "datasets>=3,<5" "accelerate>=1,<2" scikit-learn pandas matplotlib
```


In [ ]:
from datasets import Dataset, DatasetDict

ticket_splits = DatasetDict({
    name: Dataset.from_pandas(frame[["text", "label"]], preserve_index=False)
    for name, frame in [("train", train_tickets), ("validation", validation_tickets), ("test", test_tickets)]
})

from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, set_seed

BERT_CHECKPOINT = "distilbert/distilbert-base-uncased"
TICKET_TOKENIZER_CACHE_DIR = "../archive/AI_SS/session14-15/ticket-tokenizer-cache"
# Use the tokenizer paired with the pretrained encoder.
ticket_tokenizer = AutoTokenizer.from_pretrained(BERT_CHECKPOINT, cache_dir=TICKET_TOKENIZER_CACHE_DIR)
SEED = 42

MAX_TICKET_TOKENS = 96

def prepare_tickets(batch):
    return ticket_tokenizer(batch["text"], truncation=True, max_length=MAX_TICKET_TOKENS)

encoded_tickets = ticket_splits.map(prepare_tickets, batched=True, remove_columns=["text"])
set_seed(SEED)  # Initialise the new task head reproducibly.
TICKET_MODEL_CACHE_DIR = "../archive/AI_SS/session14-15/ticket-model-cache"
ticket_model = AutoModelForSequenceClassification.from_pretrained(
    BERT_CHECKPOINT, num_labels=len(class_names), cache_dir=TICKET_MODEL_CACHE_DIR,
    id2label=id_to_label, label2id=label_to_id,
)
ticket_collator = DataCollatorWithPadding(tokenizer=ticket_tokenizer)


## 4. Train and Validate

Accuracy counts all correct predictions. Macro F1 gives each category equal weight; a confusion matrix shows where errors occur. Both the encoder and classification head update here. Freezing the encoder would instead train a task head on fixed language features.

Compare against a majority-class baseline. The untrained head is a useful diagnostic but is not an already competent ticket classifier.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay
from transformers import Trainer, TrainingArguments

def ticket_metrics(prediction):
    logits, labels = prediction
    predicted = logits.argmax(axis=-1)
    return {
        "accuracy": accuracy_score(labels, predicted),
        "macro_f1": f1_score(labels, predicted, labels=list(id_to_label), average="macro", zero_division=0),
    }

TICKET_CHECKPOINT_DIR = "../archive/AI_SS/session14-15/ticket-checkpoints"
ticket_args = TrainingArguments(
    output_dir=TICKET_CHECKPOINT_DIR,
    learning_rate=3e-5, per_device_train_batch_size=8, per_device_eval_batch_size=8,
    num_train_epochs=3, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
    load_best_model_at_end=True, metric_for_best_model="macro_f1", greater_is_better=True,
    logging_steps=5, report_to="none", seed=SEED, dataloader_num_workers=0,
)
ticket_trainer = Trainer(
    model=ticket_model, args=ticket_args,
    train_dataset=encoded_tickets["train"], eval_dataset=encoded_tickets["validation"],
    processing_class=ticket_tokenizer, data_collator=ticket_collator, compute_metrics=ticket_metrics,
)
ticket_trainer.train()
print("Validation:", ticket_trainer.evaluate())


## 5. Compare Both Models on the Same Held-Out Tickets

Use identical test tickets and labels for both methods. Accuracy and macro F1 measure their predictions; confusion matrices and examples reveal different errors. The majority-class result gives a simple baseline.

The test set contains only 16 fictional tickets. A pretrained Transformer is not guaranteed to beat the smaller NN, and a few changed predictions can substantially affect these scores.


In [ ]:
import matplotlib.pyplot as plt

after_ticket = ticket_trainer.predict(encoded_tickets["test"])
true_labels = after_ticket.label_ids
predicted_labels = after_ticket.predictions.argmax(axis=-1)
X_count_test = torch.tensor(count_vectorizer.transform(test_tickets["text"]).toarray(), dtype=torch.float32)
count_model.eval()
with torch.no_grad():
    count_predictions = count_model(X_count_test).argmax(dim=1).numpy()
majority_label = int(train_tickets["label"].mode().iloc[0])
majority_predictions = [majority_label] * len(true_labels)
comparison_rows = []
for name, predictions in [("Majority class", majority_predictions), ("Word counts + NN", count_predictions), ("DistilBERT", predicted_labels)]:
    comparison_rows.append({
        "model": name, "accuracy": accuracy_score(true_labels, predictions),
        "macro_f1": f1_score(true_labels, predictions, labels=list(id_to_label), average="macro", zero_division=0),
    })
print(pd.DataFrame(comparison_rows).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, name, predictions in [(axes[0], "Word counts + NN", count_predictions), (axes[1], "DistilBERT", predicted_labels)]:
    print(name)
    print(classification_report(true_labels, predictions, labels=list(id_to_label), target_names=class_names, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(
        true_labels, predictions, labels=list(id_to_label), display_labels=class_names,
        xticks_rotation=30, cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()
ticket_results = pd.DataFrame({
    "text": test_tickets["text"].tolist(),
    "actual": [id_to_label[int(label)] for label in true_labels],
    "word_counts_nn": [id_to_label[int(label)] for label in count_predictions],
    "distilbert": [id_to_label[int(label)] for label in predicted_labels],
})
print("Tickets misclassified by either model:")
errors = (ticket_results["actual"] != ticket_results["word_counts_nn"]) | (ticket_results["actual"] != ticket_results["distilbert"])
print(ticket_results[errors].to_string(index=False))


## 6. Save, Reload and Route New Tickets

A softmax score is relative to the model's available classes. It is not a calibrated probability that the routing decision is correct. Out-of-scope tickets and ambiguous requests need a review process.

Save the count vocabulary together with the NN weights: changing the vocabulary order changes what each input column means. Save the DistilBERT tokenizer together with its model for the same reason.


In [ ]:
import os
import joblib
from transformers import pipeline

COUNT_SAVE_DIR = "../archive/AI_SS/session14-15/saved-count-nn"
os.makedirs(COUNT_SAVE_DIR, exist_ok=True)
joblib.dump(count_vectorizer, COUNT_SAVE_DIR + "/vectorizer.joblib")
torch.save({
    "state_dict": count_model.state_dict(), "input_size": X_count_train.shape[1],
    "hidden_size": COUNT_HIDDEN_UNITS, "class_names": class_names,
}, COUNT_SAVE_DIR + "/model.pt")
saved_count = torch.load(COUNT_SAVE_DIR + "/model.pt", weights_only=True)
reloaded_count_model = build_count_nn(saved_count["input_size"], saved_count["hidden_size"], len(saved_count["class_names"]))
reloaded_count_model.load_state_dict(saved_count["state_dict"])
reloaded_count_model.eval()
reloaded_vectorizer = joblib.load(COUNT_SAVE_DIR + "/vectorizer.joblib")

TICKET_SAVE_DIR = "../archive/AI_SS/session14-15/saved-ticket-classifier"
ticket_trainer.save_model(TICKET_SAVE_DIR)
ticket_tokenizer.save_pretrained(TICKET_SAVE_DIR)
ticket_classifier = pipeline("text-classification", model=TICKET_SAVE_DIR, tokenizer=TICKET_SAVE_DIR)
new_tickets = [
    "My laptop cannot connect to campus Wi-Fi.",
    "I dropped my monitor and the display is broken.",
    "The program crashes, but only when I connect to Wi-Fi.",
]
with torch.no_grad():
    new_counts = torch.tensor(reloaded_vectorizer.transform(new_tickets).toarray(), dtype=torch.float32)
    count_classes = reloaded_count_model(new_counts).argmax(dim=1).tolist()
bert_results = ticket_classifier(new_tickets, truncation=True, max_length=MAX_TICKET_TOKENS)
for text, count_class, bert_result in zip(new_tickets, count_classes, bert_results):
    print(text)
    print("Word counts + NN:", saved_count["class_names"][count_class], "| DistilBERT:", bert_result)
